# GPU tour: milling stability charts (branch `hill-argument-principle`)

Runs `hill/gpu_tour_milling.jl` on the GPU of this runtime: every milling model in every number
format (Float32; Float16 evaluation with a Float32 march; Float16 + a Float32 re-check of the
flagged points), timed at full HD and checked against a Float64 reference on a coarse grid.
Pick the GPU under *Runtime → Change runtime type*, then *Run all* (~10 min, mostly the Julia
setup). **Afterwards: *Runtime → Disconnect and delete runtime*.**


In [ ]:
REPO, BRANCH, JULIA_CHANNEL = "https://github.com/bachrathyd/InterpolatedNyquist.jl", "hill-argument-principle", "1.12"
REPO_DIR = '/content/InterpolatedNyquist.jl'
import os, subprocess

def sh(cmd):
    "Run a shell command, stream its output, and STOP on failure."
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'command failed (exit code {p.returncode}) -- see the output above')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
os.environ['PATH'] = '/root/.juliaup/bin:' + os.environ['PATH']
os.environ['JULIA_NUM_THREADS'] = 'auto'
if not os.path.exists('/root/.juliaup/bin/julia'):
    sh(f'curl -fsSL https://install.julialang.org | sh -s -- --yes --default-channel {JULIA_CHANNEL} > /dev/null')
if os.path.isdir(REPO_DIR):
    sh(f'cd {REPO_DIR} && git fetch -q --depth 1 origin {BRANCH} && git reset -q --hard FETCH_HEAD')
else:
    sh(f'git clone -q -b {BRANCH} --depth 1 {REPO} {REPO_DIR}')
sh(f"cd {REPO_DIR} && git log -1 --format='%h  %ad  %s' --date=short")
sh(f"cd {REPO_DIR} && julia --project=gpu/scripts -e 'using Pkg; Pkg.instantiate(); Pkg.precompile()' 2>&1 | tail -3")

In [ ]:
MODELS = 'mill2g,mill3h,mill3d'   # see MODELS in hill/gpu_tour_milling.jl
sh(f"cd {REPO_DIR} && julia --project=gpu/scripts hill/gpu_tour_milling.jl --models {MODELS} "
   f"--res 1920x1080 --check 192x108 --csv /content/gpu_tour.csv 2>&1")
